# Segment a scene

Semantic segmentation labels *every point* in a scene. A whole room rarely fits in one forward pass, so this library separates **what** the model computes from **how** it is run over a large scene: that second job belongs to an `Inferer`.

This notebook has two halves:

- **The inferer contract**, demonstrated on a synthetic room with a toy predictor. This runs anywhere, no model or dataset needed.
- **A real pretrained model** with its registered preprocessing, scored at full point resolution on the room committed with these docs.

New to the library? Start with the [Quickstart](01-quickstart.md).

In [ ]:
# On Colab: !pip install "torch-pointcloud[pyg-lib]"
import torch

import torch_pointcloud as tp

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch-pointcloud", tp.__version__, "| device:", device)

In [ ]:
import matplotlib.pyplot as plt


def show_cloud(pos, color=None, *, ax=None, title=None, size=6, cmap="viridis"):
    """Scatter a point cloud. `pos` is (N, 3); `color` is per-point RGB, a label vector, or None."""
    if ax is None:
        ax = plt.figure(figsize=(4, 4)).add_subplot(projection="3d")

    p = pos.detach().cpu().numpy()
    c = color.detach().cpu().numpy() if torch.is_tensor(color) else color
    kw = {} if c is None else {"cmap": cmap}
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], c=c, s=size, depthshade=False, linewidths=0, **kw)
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=10)
    return ax

## The inferer contract

An [inferer](../inferers/overview.md) takes a packed-batch dict and a `predictor` callable, and returns one prediction per input point, shape $(N, C_\text{out})$, aligned to the input order. The predictor maps a (sub-)scene dict to logits; the inferer decides whether that happens in one pass, over tiled blocks, or under test-time augmentation, and stitches the partial results together.

To see this with no model at all, we build a synthetic $10 \times 10 \times 3$ m room and use a *toy predictor* that labels points by height into four bands.

In [ ]:
room = torch.rand(40_000, 3) * torch.tensor([10.0, 10.0, 3.0])
scene = {"pos": room, "batch": torch.zeros(len(room), dtype=torch.long)}

BANDS = ["band 0", "band 1", "band 2", "band 3"]


def height_predictor(d):
    """Toy predictor: 4 'classes' by height (z), as confident one-hot logits."""
    z = d["pos"][:, 2]
    band = (z / 3.0 * 4).clamp(0, 3).long()
    return torch.nn.functional.one_hot(band, num_classes=4).float() * 5.0

The simplest inferer just calls the predictor on the whole scene:

In [ ]:
from torch_pointcloud.inferers import SimpleInferer

whole = SimpleInferer()(scene, predictor=height_predictor)
print("output:", tuple(whole.shape), "| aligned to pos:", whole.shape[0] == scene["pos"].shape[0])

When the scene is too large for one pass, `SlidingWindowInferer` tiles it into cubic blocks of a fixed metric size, runs the predictor on each, and blends overlapping predictions. With `mode="gaussian"` points near a block center count more than those at the seam, which softens block boundaries.

Blocks are spaced `block_size * (1 - overlap)` apart, so most points fall in several blocks and the seams average out.

The output is still one row per original point, so the two runs can be colored side by side:

In [ ]:
from torch_pointcloud.inferers import SlidingWindowInferer

inferer = SlidingWindowInferer(block_size=3.0, overlap=0.5, mode="gaussian")
tiled = inferer(scene, predictor=height_predictor)

print("output:", tuple(tiled.shape))
print("identical to the whole-scene labels:", bool((tiled.argmax(-1) == whole.argmax(-1)).all()))

fig = plt.figure(figsize=(9, 4))
show_cloud(room, color=whole.argmax(-1), ax=fig.add_subplot(121, projection="3d"), title="whole scene at once", size=1)
show_cloud(room, color=tiled.argmax(-1), ax=fig.add_subplot(122, projection="3d"), title="stitched from 3 m blocks", size=1);

![The synthetic room labeled by the toy predictor in one pass, beside the same room stitched from 3 m blocks, the two indistinguishable.](../assets/tutorials/segmentation_inferer.png)

The two panels are the same picture. A 3 m block holds 3,507 of the 40,000 points, so the predictor never sees more than a tenth of the scene at once, yet blending the blocks back together reproduces the whole-scene labels on every point. With a predictor this simple the stitching is lossless. Both panels are drawn nearly side on and from a 12,000-point subsample, so the four height bands read as stacked stripes instead of one solid block.

Test-time augmentation wraps *any* base inferer: it runs several augmented passes and averages them, without touching the predictor.

```python
from torch_pointcloud.inferers import TTAInferer
from torch_pointcloud.transforms import Compose, RandomFlip, RandomRotate

tta = TTAInferer(
    base=SlidingWindowInferer(block_size=3.0, overlap=0.5),
    transforms=Compose([
        RandomRotate(keys="pos", angle_range=(-180.0, 180.0), axis=2, p=1.0),
        RandomFlip(keys="pos", axes=[0, 1], p=0.5),
    ]),
    num_passes=4,
)
probs = tta(scene, predictor=height_predictor)
```

`KNNWindowInferer` and `VoxelPartitionInferer` are two more strategies with the same contract. The [Inferers guide](../inferers/overview.md) compares them.

## A real pretrained model

Voxel backbones (SpUNet, SPVCNN, PT-V3) do not consume raw points directly: they voxelize the cloud, run sparse convolutions on the grid, and map predictions back. All of that lives in the **transform pipeline the checkpoint was trained with**, which `create_model(..., return_info=True)` returns alongside the model.

> The rest of the notebook needs the `spconv` extra and the checkpoint (157 MB, downloaded to a local cache on first use). It runs on one room committed with these docs; `ScanNet20(root="data", split="val")[0]` swaps in the benchmark instead.

In [ ]:
model, info = tp.create_model(
    "spunet-v1m1.scannet20.pointcept",
    task="segmentation",
    pretrained=True,
    return_info=True,
)
model = model.eval().to(device)
classes = list(info["weights"]["classes"])
print(len(classes), "classes:", ", ".join(classes))

The room is a ScanNet scene kept as the vertices of its reconstructed mesh, with a color and a NYU40 semantic id per vertex. Two things are missing before the checkpoint can read it.

**Normals.** The ScanNet20 checkpoints take six features per point, $[r, g, b, n_x, n_y, n_z]$, and the file carries no normals, so estimate them from local geometry with `EstimateNormals`.

**The benchmark's label ordering.** The file's `segment` holds raw NYU40 ids, which agree with the 20-class benchmark up to `counter` and diverge after it: a refrigerator is $24$ in NYU40 and $15$ in the benchmark. `SCANNET20_LABELS` publishes the mapping, so one `Relabel` bridges the two.

In [ ]:
import urllib.request
from pathlib import Path

import numpy as np
from plyfile import PlyData

import torch_pointcloud.transforms as T
from torch_pointcloud.datasets.scannet import SCANNET20_LABELS

path = Path("../assets/data/sample_scene_labeled.ply")  # in a docs checkout
if not path.exists():
    path = Path("sample_scene_labeled.ply")
    url = "https://github.com/arthurdjn/pytorch-pointcloud/raw/main/docs/assets/data/sample_scene_labeled.ply"
    if not path.exists():
        urllib.request.urlretrieve(url, path)

vertex = PlyData.read(path)["vertex"]
room = {
    "pos": torch.from_numpy(np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1).astype(np.float32)),
    "color": torch.from_numpy(np.stack([vertex["red"], vertex["green"], vertex["blue"]], axis=1).astype(np.float32)),
    "segment": torch.from_numpy(np.asarray(vertex["segment"]).astype(np.int64)),
}
room = T.EstimateNormals(keys="pos", k=16)(room)
print("points:", len(room["pos"]), "| normals:", tuple(room["normal"].shape))

print("NYU40 ids in the room:", sorted(set(room["segment"].tolist())))
room = T.Relabel(keys="segment", labels=SCANNET20_LABELS)(room)
print("after Relabel:      ", sorted(set(room["segment"].tolist())))

The checkpoint's own pipeline finishes the job: it shifts those ids once more, onto the $0..19$ the model predicts, and marks everything outside the twenty classes as $-1$. It also voxelizes the room at 2 cm, builds `x` from color and normal, and writes an `inverse` map from every original point to the voxel it landed in. It also copies the labels to `origin_segment` *before* voxelizing, which is what makes a full-resolution score possible. `collate` then adds the packed `batch` index.

In [ ]:
from torch_pointcloud.utils.data import collate

batch = collate([info["transform"]({key: value.clone() for key, value in room.items()})])
print({key: tuple(value.shape) for key, value in batch.items() if torch.is_tensor(value)})

The room fits in one forward pass, so `SimpleInferer` is enough, and the predictor is where the voxel logits are broadcast back to the original points through `inverse`. It is the same contract as the toy predictor above: a scene dict in, one row per input point out. A scene that did not fit would swap in `SlidingWindowInferer` with no other change.

In [ ]:
def predictor(data):
    """Voxel logits, broadcast back to one row per original point through the `inverse` map."""
    logits = model(data["x"].to(device), data["pos_grid"].to(device), data["batch"].to(device))
    return logits[data["inverse"].to(device)]


with torch.no_grad():
    labels = SimpleInferer(softmax=True)(batch, predictor=predictor).argmax(dim=-1).cpu()

print("one row per input point:", len(labels) == len(room["pos"]))

Score it against `origin_segment`, the room's own annotation at full resolution. The points the room carries no label for stay out of both metrics through `ignore_index=-1`.

In [ ]:
from torch_pointcloud.utils.metrics import confusion_matrix

target = batch["origin_segment"]
annotated = target >= 0
matrix = confusion_matrix(labels, target, model.num_classes, ignore_index=-1)
union = matrix.sum(0) + matrix.sum(1) - matrix.diag()
iou = matrix.diag() / union.clamp_min(1)

print(f"annotated points: {int(annotated.sum())} of {len(target)}")
print(f"accuracy: {(labels[annotated] == target[annotated]).float().mean():.3f}")
print(f"mIoU over the {int((union > 0).sum())} classes involved: {iou[union > 0].mean():.3f}")

To see *where* the model is wrong, color the annotated points by whether they came back right, and put the per-class IoU next to them:

In [ ]:
wrong = labels[annotated] != target[annotated]
ranked = torch.nonzero(union > 0).flatten()
ranked = ranked[iou[ranked].argsort()]

fig = plt.figure(figsize=(10, 4))
show_cloud(
    room["pos"][annotated],
    color=wrong.long(),
    ax=fig.add_subplot(121, projection="3d"),
    title=f"{int(wrong.sum()):,} misclassified points",
    size=0.4,
    cmap="coolwarm",
)
ax = fig.add_subplot(122)
ax.barh([classes[int(index)] for index in ranked], iou[ranked].tolist(), color="tab:orange")
ax.set_xlabel("IoU", fontsize=9)
ax.tick_params(labelsize=8);

![The committed room with its misclassified points picked out in red, beside a bar chart of per-class IoU running from floor at 0.97 down to five classes at zero.](../assets/tutorials/segmentation_room_errors.png)

80% of the annotated points are right, at a mIoU of 0.416 over the 15 classes involved. The two numbers disagree because a per-scene mIoU divides by whichever classes appear, so one class missed entirely costs it far more than it costs accuracy. Five of the fifteen bars sit at exactly zero, which is where the mIoU goes.

The red in the left panel is not spread evenly: the floor and the chairs come back almost clean, and the errors bunch on the walls and on the flat surfaces mounted against them. The pale points are the ones the room carries no label for, which the score leaves out through `ignore_index=-1`.

Which class went *where* is what the confusion matrix already computed above answers. Divide each row by its own support and it reads as the fraction of a true class predicted as the column class. A class this room has no annotated points for has nothing to normalize, so keep the classes with support and print the one prediction each of them draws most:

In [ ]:
share = matrix / matrix.sum(dim=1, keepdim=True).clamp_min(1)
for index in torch.nonzero(matrix.sum(dim=1) > 0).flatten():
    answer = int(share[index].argmax())
    print(f"{classes[index]:>15} -> {classes[answer]:<15} {share[index, answer]:.2f}")

The errors nearly all point one way. `floor` and `door` come back at 0.98 and `chair` at 0.92, but the room's refrigerator is called `wall` point for point at 1.00, 0.80 of the cabinet goes the same way, and half the sink is called `cabinet`. Flat surfaces mounted on a wall get absorbed into it, a failure a single mIoU number hides, so read the per-class breakdown before judging a checkpoint.

## Recap

- An inferer maps `(scene dict, predictor) -> per-point logits`, aligned to the input.
- For scenes that fit, run the model once and (for voxel models) broadcast back through `inverse`.
- For scenes that do not fit, tile with `SlidingWindowInferer`, optionally wrapped in `TTAInferer`.
- Score at raw resolution against `origin_segment`, and read the per-class breakdown, not just the mean.